In [25]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler , PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline


In [26]:
dataset = fetch_california_housing(as_frame = True)
df = dataset.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [27]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

print(X.shape)
print(y.shape)

(20640, 8)
(20640,)


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Multiple Linear Regression

In [29]:
model = LinearRegression()
model.fit(X_train, y_train)

train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

Train R²: 0.6125511913966952
Test R²: 0.5757877060324508


# Standardize features and fit again

In [30]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_model = LinearRegression()
scaled_model.fit(X_train_scaled, y_train)

weights = pd.Series(
    scaled_model.coef_,
    index=X.columns
).sort_values()

print("\nb) Standardized Feature Weights")
print(weights)

print("Most negative weight:", weights.index[0], weights.iloc[0])
print("Biggest positive weight:", weights.index[-1], weights.iloc[-1])



b) Standardized Feature Weights
Latitude     -0.896929
Longitude    -0.869842
AveRooms     -0.294410
AveOccup     -0.040829
Population   -0.002308
HouseAge      0.122546
AveBedrms     0.339259
MedInc        0.854383
dtype: float64
Most negative weight: Latitude -0.896928876638665
Biggest positive weight: MedInc 0.8543830309268546


In [31]:
for degree in [1, 2, 3]:
    poly_model = make_pipeline(
        PolynomialFeatures(degree=degree),
        StandardScaler(),
        LinearRegression()
    )

    poly_model.fit(X_train, y_train)

    train_score = poly_model.score(X_train, y_train)
    test_score = poly_model.score(X_test, y_test)

    poly_features = poly_model.named_steps["polynomialfeatures"]
    num_features = poly_features.n_output_features_

    print(f"\nDegree: {degree}")
    print("Number of features:", num_features)
    print("Train R²:", train_score)
    print("Test R²:", test_score)



Degree: 1
Number of features: 9
Train R²: 0.6125511913966952
Test R²: 0.575787706032451

Degree: 2
Number of features: 45
Train R²: 0.6852681982344953
Test R²: 0.6456819729261978

Degree: 3
Number of features: 165
Train R²: 0.7441415681335486
Test R²: -18.388708058555284
